In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

plt.rcParams["figure.dpi"] = 150
plt.rcParams["font.size"] = 11

C = {
    "input":   "#E8F5E9",
    "process": "#BBDEFB",
    "model":   "#C8E6C9",
    "fusion":  "#FFE0B2",
    "output":  "#FFCDD2",
    "data":    "#FFF9C4",
    "arrow":   "#546E7A",
    "text":    "#2C3E50",
}


In [ ]:
def block(ax, x, y, w, h, label, sub="", color="#BBDEFB"):
    box = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.08",
                         facecolor=color, edgecolor="#37474F", linewidth=1.8, zorder=2)
    ax.add_patch(box)
    ax.text(x + w/2, y + h*0.62, label, ha="center", va="center",
            fontsize=9, fontweight="bold", color=C["text"], zorder=3)
    if sub:
        ax.text(x + w/2, y + h*0.28, sub, ha="center", va="center",
                fontsize=6.5, color="#546E7A", zorder=3)

def arrow(ax, x1, y1, x2, y2, color="#546E7A"):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="->", color=color, lw=2), zorder=1)


In [ ]:
fig, ax = plt.subplots(figsize=(22, 9))
ax.set_title("Hệ thống Nhận diện Cảm xúc — Sơ đồ Pipeline", fontsize=16, fontweight="bold", pad=20)

Y1 = 5.5
BW, BH = 2.0, 1.4
GAP = 0.6

# === ĐẦU VÀO ===
block(ax, 0.3, Y1-0.3, 0.8, 0.7, "Webcam", "live", C["input"])
block(ax, 0.3, Y1+0.6, 0.8, 0.7, "Tải ảnh", "JPEG/PNG", C["input"])
ax.text(0.7, Y1+2.1, "ĐẦU VÀO", ha="center", fontsize=10, fontweight="bold", color="#2C3E50")
arrow(ax, 1.1, Y1+0.05, 1.8, Y1)
arrow(ax, 1.1, Y1+0.95, 1.8, Y1)

# === PHÁT HIỆN KHUÔN MẶT ===
x1 = 1.8
block(ax, x1, Y1-BH/2, BW, BH, "Phát hiện khuôn mặt", "YOLOv8 (tự train) | Haar (dự phòng)", C["process"])
arrow(ax, x1+BW, Y1, x1+BW+GAP, Y1)

# === TIỀN XỬ LÝ ===
x2 = x1 + BW + GAP
block(ax, x2, Y1-BH/2, BW, BH, "Tiền xử lý", "Cắt khuôn mặt + Resize + Chuẩn hóa", C["process"])
arrow(ax, x2+BW, Y1, x2+BW+GAP, Y1)

# === SUY LUẬN (3 models) ===
x3 = x2 + BW + GAP
block(ax, x3, Y1-BH/2, BW, BH+0.6, "Suy luận", "Chọn 1 trong 3 model", C["model"])
models = ["Baseline CNN (TF, 670K)", "DAN (Torch, 11M)", "POSTER (Torch, 58M)"]
for i, lbl in enumerate(models):
    sy = Y1-BH/2-0.3-(i+1)*0.65
    block(ax, x3+0.1, sy, BW-0.2, 0.55, lbl, "", C["model"])
    arrow(ax, x3+BW/2, Y1-BH/2-0.05, x3+BW/2, sy+0.55)
arrow(ax, x3+BW, Y1, x3+BW+GAP, Y1)

# === LÀM MỊN CẢM XÚC ===
x4 = x3 + BW + GAP
block(ax, x4, Y1-BH/2, BW, BH, "Làm mịn cảm xúc", "EMA (alpha=0.35) + giữ 3 khung", C["fusion"])
arrow(ax, x4+BW, Y1, x4+BW+GAP, Y1)

# === KẾT QUẢ ===
x5 = x4 + BW + GAP
block(ax, x5, Y1-BH/2, BW, BH+0.6, "Kết quả", "JSON: emotion, conf, bbox", C["output"])
outputs = ["Hiển thị UI (Webcam)", "Lưu Database (SQLite)", "Lưu ảnh tải lên"]
for i, lbl in enumerate(outputs):
    oy = Y1-BH/2-0.3-(i+1)*0.65
    block(ax, x5+0.1, oy, BW-0.2, 0.55, lbl, "", C["output"])
    arrow(ax, x5+BW/2, Y1-BH/2-0.05, x5+BW/2, oy+0.55)

# === QUY TRÌNH HUẤN LUYỆN ===
ax.axhline(y=0.3, xmin=0, xmax=0.7, color="#90A4AE", lw=1.5, ls="--", zorder=0)
ax.text(10, -0.05, "QUY TRÌNH HUẤN LUYỆN", ha="center", fontsize=11, fontweight="bold", color="#546E7A", style="italic")
TY = -0.4
TBW, TBH = 1.8, 1.0
TGAP = 0.5
train = [
    (0.5, TY, "Bộ dữ liệu RAF-DB", "15.339 ảnh, 7 cảm xúc", C["data"]),
    (0.5+TBW+TGAP, TY, "Tiền xử lý", "Resize + chuẩn hóa", C["data"]),
    (0.5+2*(TBW+TGAP), TY, "Tăng cường dữ liệu", "Flip, brightness, contrast...", C["data"]),
    (0.5+3*(TBW+TGAP), TY, "Huấn luyện (Colab)", "GPU: DAN/POSTER | CPU: CNN", C["process"]),
    (0.5+4*(TBW+TGAP), TY, "Lưu Checkpoint", ".keras / .pth", C["process"]),
    (0.5+5*(TBW+TGAP), TY, "Nạp vào Mô hình", "Inference server khi khởi động", C["model"]),
]
for i, (x, y, lbl, sub, c) in enumerate(train):
    block(ax, x, y-TBH/2, TBW, TBH, lbl, sub, c)
    if i < len(train)-1:
        arrow(ax, x+TBW, y, train[i+1][0], y)

ckpt_x = train[4][0]
arrow(ax, ckpt_x+TBW/2, TY+TBH/2, x3+BW/2, Y1-BH/2, "#90A4AE")
ax.text(ckpt_x+0.5, (TY+TBH/2 + Y1-BH/2)/2, "tải trọng số", fontsize=7, color="#90A4AE", style="italic", rotation=90)

ax.set_xlim(0, 22)
ax.set_ylim(-1.2, 8)
ax.axis("off")
plt.tight_layout()
plt.savefig("system_pipeline.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: system_pipeline.png")
